# 02 - SQLite and Profiling

Take the cleaned tables from notebook 01 and:

1. write each uploaded file into its own SQLite database: a `cleaned`
   table with proper column types and a `<name>_raw` table with the
   original text values;
2. register the statistics SQLite lacks (median, standard deviation,
   percentile) as plain functions;
3. profile every table (EDA): ranges, missing values, top categories,
   correlations, plus the flags from cleaning;
4. build the compact **schema text** the model will see in notebook 04.

No model is used here.

Becomes `src/helpers/sqlite_store.py` and `src/helpers/data_profiler.py`.

In [1]:
# ============================================================
# 1. CONFIGURATION
# ============================================================

import json
import logging
import math
import sqlite3
import statistics
from pathlib import Path

import pandas as pd


BASE_DIR = (
    Path.cwd().parent if Path.cwd().name == "notebook" else Path.cwd()
)
CLEANED_DIR = BASE_DIR / "data" / "output_data" / "cleaned_data"
DATASETS_DIR = BASE_DIR / "data" / "output_data" / "datasets"

DATABASE_NAME = "dataset.db"
PROFILE_NAME = "profile.json"
SCHEMA_NAME = "schema.txt"
REPORT_NAME = "cleaning_report.json"

RAW_TABLE_SUFFIX = "_raw"
INSERT_BATCH_SIZE = 5000

# SQLite column type for each kind found by notebook 01.
SQLITE_TYPES = {
    "integer": "INTEGER",
    "float": "REAL",
    "boolean": "INTEGER",
    "date": "TEXT",
    "datetime": "TEXT",
    "text": "TEXT",
}

# Profiling
TOP_VALUES = 5
CATEGORY_MAX_UNIQUE_RATIO = 0.5      # same rule as notebook 01
CORRELATION_MIN_ABS = 0.5
CORRELATION_MIN_ROWS = 5

# Schema text for the model
SCHEMA_ALL_VALUES_LIMIT = 15         # list every category up to this many
SCHEMA_SAMPLE_VALUES = 3             # examples for other text columns

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(message)s",
    force=True,
)
logger = logging.getLogger("02_sqlite_profiling")

DATASETS_DIR.mkdir(parents=True, exist_ok=True)
print("Cleaned tables:", CLEANED_DIR)
print("Datasets      :", DATASETS_DIR)
print("SQLite version:", sqlite3.sqlite_version)

Cleaned tables: G:\GEN AI CE2\data\output_data\cleaned_data
Datasets      : G:\GEN AI CE2\data\output_data\datasets
SQLite version: 3.50.4


## 2. Loading notebook 01 outputs

Tables are grouped by the file they came from: one uploaded file is one
dataset (an Excel file with two sheets gives one dataset, two tables).

In [2]:
# ============================================================
# 2. LOADING NOTEBOOK 01 OUTPUTS
# ============================================================

def load_cleaned_tables(cleaned_dir):
    """
    Read every cleaned table, its raw copy and its report.

    Returns:
        dict: source file -> list of {"table_name", "cleaned", "raw",
              "report"}
    """

    datasets = {}

    for report_path in sorted(cleaned_dir.glob("*_report.json")):
        with report_path.open(encoding="utf-8") as file:
            report = json.load(file)

        name = report["table_name"]
        datasets.setdefault(report["source_file"], []).append({
            "table_name": name,
            "cleaned": pd.read_parquet(cleaned_dir / f"{name}.parquet"),
            "raw": pd.read_parquet(cleaned_dir / f"{name}_raw.parquet"),
            "report": report,
        })

    return datasets


datasets = load_cleaned_tables(CLEANED_DIR)

for source_file, tables in datasets.items():
    print(f"{source_file:<22} -> {[table['table_name'] for table in tables]}")

customers.json         -> ['customers']
messy_employees.xlsx   -> ['messy_employees_employees', 'messy_employees_targets']
messy_sales.csv        -> ['messy_sales']
products.parquet       -> ['products']


## 3. Writing to SQLite

SQLite has no date or boolean type, so:

- dates are stored as ISO text `YYYY-MM-DD` (`datetime`: `YYYY-MM-DD HH:MM:SS`),
  which sorts correctly and works with `strftime`;
- booleans are stored as `1` / `0`.

Values are converted column by column (fast), then inserted in batches.

In [3]:
# ============================================================
# 3. WRITING TO SQLITE
# ============================================================

def quote_identifier(name):
    """Quote a table or column name for SQL."""

    return '"' + str(name).replace('"', '""') + '"'


def to_sqlite_column(series, kind):
    """Convert one column to plain Python values SQLite accepts."""

    if kind == "date":
        values = series.dt.strftime("%Y-%m-%d")
    elif kind == "datetime":
        values = series.dt.strftime("%Y-%m-%d %H:%M:%S")
    elif kind == "boolean":
        values = series.astype("Int64")
    elif kind == "integer":
        values = series.astype("Int64")
    elif kind == "float":
        values = series.astype("Float64")
    else:
        values = series.astype("string")

    # Missing values of every kind (NA, NaT, NaN) become None.
    values = values.astype(object)

    return values.where(values.notna(), None)


def create_table(connection, table_name, column_types):
    """Drop and recreate one table with explicit column types."""

    columns_sql = ",\n    ".join(
        f"{quote_identifier(column)} {SQLITE_TYPES[kind]}"
        for column, kind in column_types.items()
    )

    connection.execute(f"DROP TABLE IF EXISTS {quote_identifier(table_name)}")
    connection.execute(
        f"CREATE TABLE {quote_identifier(table_name)} (\n    {columns_sql}\n)"
    )


def insert_rows(connection, table_name, frame, column_types):
    """Insert a DataFrame in batches. Returns the number of rows."""

    columns = list(column_types)
    prepared = pd.DataFrame({
        column: to_sqlite_column(frame[column], column_types[column])
        for column in columns
    })

    placeholders = ", ".join("?" for _ in columns)
    column_sql = ", ".join(quote_identifier(column) for column in columns)
    sql = (f"INSERT INTO {quote_identifier(table_name)} ({column_sql}) "
           f"VALUES ({placeholders})")

    rows = prepared.values.tolist()

    for start in range(0, len(rows), INSERT_BATCH_SIZE):
        connection.executemany(sql, rows[start:start + INSERT_BATCH_SIZE])

    return len(rows)


def write_dataset(database_path, tables):
    """
    Write every table of one dataset (cleaned + raw) into SQLite.

    Returns:
        dict: table name -> {"rows", "raw_table", "raw_rows"}
    """

    database_path.parent.mkdir(parents=True, exist_ok=True)
    written = {}

    with sqlite3.connect(database_path) as connection:
        for table in tables:
            name = table["table_name"]
            raw_name = f"{name}{RAW_TABLE_SUFFIX}"
            column_types = table["report"]["column_types"]

            # Step 1: cleaned table with real types.
            create_table(connection, name, column_types)
            rows = insert_rows(connection, name, table["cleaned"],
                               column_types)

            # Step 2: raw table, every column text.
            raw_types = {column: "text" for column in table["raw"].columns}
            create_table(connection, raw_name, raw_types)
            raw_rows = insert_rows(connection, raw_name, table["raw"],
                                   raw_types)

            written[name] = {"rows": rows, "raw_table": raw_name,
                             "raw_rows": raw_rows}

            logger.info("Wrote %s (%d rows) and %s (%d rows) to %s",
                        name, rows, raw_name, raw_rows, database_path.name)

    return written


def dataset_folder(source_file):
    """messy_sales.csv -> data/output_data/datasets/messy_sales/"""

    return DATASETS_DIR / Path(source_file).stem


written_tables = {}

for source_file, tables in datasets.items():
    database_path = dataset_folder(source_file) / DATABASE_NAME
    written_tables[source_file] = write_dataset(database_path, tables)

written_tables

2026-11-14 19:25:55,249 | INFO | Wrote customers (8 rows) and customers_raw (8 rows) to dataset.db


2026-11-14 19:25:55,278 | INFO | Wrote messy_employees_employees (20 rows) and messy_employees_employees_raw (20 rows) to dataset.db


2026-11-14 19:25:55,281 | INFO | Wrote messy_employees_targets (4 rows) and messy_employees_targets_raw (4 rows) to dataset.db


2026-11-14 19:25:55,318 | INFO | Wrote messy_sales (32 rows) and messy_sales_raw (35 rows) to dataset.db


2026-11-14 19:25:55,344 | INFO | Wrote products (5 rows) and products_raw (5 rows) to dataset.db


{'customers.json': {'customers': {'rows': 8,
   'raw_table': 'customers_raw',
   'raw_rows': 8}},
 'messy_employees.xlsx': {'messy_employees_employees': {'rows': 20,
   'raw_table': 'messy_employees_employees_raw',
   'raw_rows': 20},
  'messy_employees_targets': {'rows': 4,
   'raw_table': 'messy_employees_targets_raw',
   'raw_rows': 4}},
 'messy_sales.csv': {'messy_sales': {'rows': 32,
   'raw_table': 'messy_sales_raw',
   'raw_rows': 35}},
 'products.parquet': {'products': {'rows': 5,
   'raw_table': 'products_raw',
   'raw_rows': 5}}}

## 4. Statistics SQLite does not have

SQLite has `AVG`, `MIN`, `MAX`, `SUM`, `COUNT` and math functions such
as `SQRT`, but no median, standard deviation or percentile. Proper SQLite
aggregates need a Python class, so instead these are plain functions
that take a JSON array, which `json_group_array()` builds:

```sql
SELECT city, median_of(json_group_array(total_sales)) AS median_sales
FROM messy_sales
GROUP BY city;
```

`register_functions(connection)` must run on every connection the agent
uses (notebook 05 does the same).

In [4]:
# ============================================================
# 4. STATISTICS SQLITE DOES NOT HAVE
# ============================================================

def json_numbers(json_array):
    """'[1, null, 3]' -> [1.0, 3.0]; anything unusable is skipped."""

    if json_array is None:
        return []

    try:
        values = json.loads(json_array)

    except (TypeError, ValueError):
        return []

    numbers = []

    for value in values if isinstance(values, list) else []:
        if isinstance(value, bool) or value is None:
            continue

        try:
            number = float(value)

        except (TypeError, ValueError):
            continue

        if not math.isnan(number):
            numbers.append(number)

    return numbers


def median_of(json_array):
    numbers = json_numbers(json_array)
    return statistics.median(numbers) if numbers else None


def stdev_of(json_array):
    """Sample standard deviation (needs at least two values)."""

    numbers = json_numbers(json_array)
    return statistics.stdev(numbers) if len(numbers) > 1 else None


def variance_of(json_array):
    numbers = json_numbers(json_array)
    return statistics.variance(numbers) if len(numbers) > 1 else None


def percentile_of(json_array, percent):
    """Linear-interpolated percentile, percent between 0 and 100."""

    numbers = sorted(json_numbers(json_array))

    if not numbers or percent is None or not 0 <= percent <= 100:
        return None

    position = (len(numbers) - 1) * percent / 100
    lower = math.floor(position)
    upper = math.ceil(position)

    return numbers[lower] + (numbers[upper] - numbers[lower]) * (position - lower)


SQL_FUNCTIONS = {
    "median_of": (median_of, 1),
    "stdev_of": (stdev_of, 1),
    "variance_of": (variance_of, 1),
    "percentile_of": (percentile_of, 2),
}


def register_functions(connection):
    """Add the statistics functions to one SQLite connection."""

    for name, (function, arg_count) in SQL_FUNCTIONS.items():
        connection.create_function(name, arg_count, function,
                                   deterministic=True)

    return connection


sales_db = dataset_folder("messy_sales.csv") / DATABASE_NAME

with sqlite3.connect(sales_db) as connection:
    register_functions(connection)
    result = pd.read_sql_query(
        """
        SELECT city,
               COUNT(*)                                   AS orders,
               ROUND(AVG(total_sales), 2)                 AS mean_sales,
               median_of(json_group_array(total_sales))   AS median_sales,
               ROUND(stdev_of(json_group_array(total_sales)), 2) AS stdev_sales
        FROM messy_sales
        GROUP BY city
        ORDER BY orders DESC
        """,
        connection,
    )

result

,city,orders,mean_sales,median_sales,stdev_sales
0,Delhi,14,805358.86,27750.0,2651046.19
1,New Delhi,7,52404.29,21000.0,87258.15
2,Mumbai,4,126650.00,43500.0,198015.71
3,Bengaluru,3,195083.33,108000.0,243593.45
4,Pune,2,129750.00,129750.0,166523.65
5,Mumbay,2,63900.00,63900.0,22486.00


## 5. Profiling (EDA)

One profile per table, built from the cleaned DataFrame and its cleaning
report. Everything is plain JSON so the API can return it as is.

In [5]:
# ============================================================
# 5. PROFILING (EDA)
# ============================================================

def to_plain(value):
    """numpy / pandas scalars -> JSON-safe Python values."""

    if value is None:
        return None

    if isinstance(value, pd.Timestamp):
        return None if pd.isna(value) else value.isoformat()

    if hasattr(value, "item"):
        value = value.item()

    if isinstance(value, float):
        if math.isnan(value):
            return None
        return int(value) if value.is_integer() else round(value, 4)

    return value


def profile_numeric(series):
    values = series.dropna().astype(float)

    if values.empty:
        return {}

    return {
        "min": to_plain(values.min()),
        "max": to_plain(values.max()),
        "mean": to_plain(values.mean()),
        "median": to_plain(values.median()),
        "std": to_plain(values.std()) if len(values) > 1 else None,
        "p25": to_plain(values.quantile(0.25)),
        "p75": to_plain(values.quantile(0.75)),
        "sum": to_plain(values.sum()),
    }


def profile_date(series):
    values = series.dropna()

    if values.empty:
        return {}

    return {
        "min": values.min().strftime("%Y-%m-%d"),
        "max": values.max().strftime("%Y-%m-%d"),
        "span_days": int((values.max() - values.min()).days),
    }


def profile_boolean(series):
    values = series.dropna()

    return {"true": int(values.sum()), "false": int((~values).sum())}


def profile_text(series):
    values = series.dropna().astype(str)

    if values.empty:
        return {}

    counts = values.value_counts()

    return {
        "is_category": bool(
            values.nunique() <= CATEGORY_MAX_UNIQUE_RATIO * len(values)
        ),
        "is_unique": bool(values.nunique() == len(values)),
        "top_values": [[value, int(count)]
                       for value, count in counts.head(TOP_VALUES).items()],
        "all_values": sorted(counts.index)
        if len(counts) <= SCHEMA_ALL_VALUES_LIMIT else None,
        "examples": list(dict.fromkeys(values))[:SCHEMA_SAMPLE_VALUES],
        "avg_length": to_plain(values.str.len().mean()),
    }


def profile_column(series, kind, unit):
    """Profile one column according to its kind."""

    missing = int(series.isna().sum())
    profile = {
        "kind": kind,
        "sqlite_type": SQLITE_TYPES[kind],
        "unit": unit,
        "missing": missing,
        "missing_pct": round(100 * missing / len(series), 1) if len(series) else 0,
        "unique": int(series.nunique(dropna=True)),
    }

    if kind in {"integer", "float"}:
        profile.update(profile_numeric(series))
    elif kind in {"date", "datetime"}:
        profile.update(profile_date(series))
    elif kind == "boolean":
        profile.update(profile_boolean(series))
    else:
        profile.update(profile_text(series))

    return profile


def find_correlations(frame, column_types):
    """Pairs of numeric columns with |Pearson r| >= CORRELATION_MIN_ABS."""

    numeric = [column for column, kind in column_types.items()
               if kind in {"integer", "float"}]

    if len(numeric) < 2:
        return []

    matrix = frame[numeric].astype(float).corr(min_periods=CORRELATION_MIN_ROWS)
    pairs = []

    for position, first in enumerate(numeric):
        for second in numeric[position + 1:]:
            value = matrix.loc[first, second]

            if pd.notna(value) and abs(value) >= CORRELATION_MIN_ABS:
                pairs.append({"columns": [first, second],
                              "r": round(float(value), 3)})

    return sorted(pairs, key=lambda pair: abs(pair["r"]), reverse=True)


def profile_table(table, raw_table_name):
    """Full profile for one cleaned table."""

    frame = table["cleaned"]
    report = table["report"]
    column_types = report["column_types"]
    units = report.get("column_units", {})

    return {
        "table_name": table["table_name"],
        "raw_table": raw_table_name,
        "source_file": report["source_file"],
        "rows": len(frame),
        "columns": {
            column: profile_column(frame[column], kind, units.get(column))
            for column, kind in column_types.items()
        },
        "correlations": find_correlations(frame, column_types),
        "flags": report["flags"],
        "cleaning": {
            "rows_before": report["rows_before"],
            "rows_after": report["rows_after"],
            "actions": report["actions"],
        },
    }


def profile_dataset(tables, written):
    """Profile every table of one dataset."""

    return {
        "tables": {
            table["table_name"]: profile_table(
                table, written[table["table_name"]]["raw_table"]
            )
            for table in tables
        }
    }


profiles = {
    source_file: profile_dataset(tables, written_tables[source_file])
    for source_file, tables in datasets.items()
}

print(json.dumps(
    profiles["messy_sales.csv"]["tables"]["messy_sales"]["columns"]["total_sales"],
    indent=2,
))
print(json.dumps(
    profiles["messy_sales.csv"]["tables"]["messy_sales"]["correlations"],
    indent=2,
))

{
  "kind": "integer",
  "sqlite_type": "INTEGER",
  "unit": "currency",
  "missing": 0,
  "missing_pct": 0.0,
  "unique": 25,
  "min": 1600,
  "max": 9999999,
  "mean": 410031.375,
  "median": 27750,
  "std": 1756229.77,
  "p25": 9450,
  "p75": 108000,
  "sum": 13121004
}
[
  {
    "columns": [
      "units_sold",
      "discount_pct"
    ],
    "r": 0.523
  }
]


## 6. Schema text for the model

A small model does best with a short, exact description: column types,
real category values (SQLite `=` is case-sensitive, so the model must use
`'Delhi'`, not `'delhi'`), ranges, units and the data warnings. Raw
tables are left out; the model only queries cleaned tables.

In [6]:
# ============================================================
# 6. SCHEMA TEXT FOR THE MODEL
# ============================================================

def format_number(value):
    """No thousands separators: the model must not copy commas into SQL."""

    if value is None:
        return "?"
    if isinstance(value, float):
        return f"{value:.2f}"
    return str(value)


def describe_column(name, column):
    """One schema line, e.g. city TEXT  category: 'Delhi', 'Pune'"""

    kind = column["kind"]
    parts = [f"{name} {column['sqlite_type']}"]

    if kind in {"integer", "float"}:
        if column["unit"] == "percent":
            parts.append("percent (10 means 10%)")
        elif column["unit"] == "currency":
            parts.append("currency amount")
        parts.append(f"range {format_number(column.get('min'))} to "
                     f"{format_number(column.get('max'))}")

    elif kind == "date":
        parts.append(f"date 'YYYY-MM-DD', {column.get('min')} to "
                     f"{column.get('max')}")

    elif kind == "datetime":
        parts.append(f"datetime 'YYYY-MM-DD HH:MM:SS', {column.get('min')} "
                     f"to {column.get('max')}")

    elif kind == "boolean":
        parts.append("boolean stored as 1 = true, 0 = false")

    else:
        # Unique and never missing: a key the model can join or count on.
        if column.get("is_unique") and not column["missing"]:
            parts.append("unique per row")

        if column.get("all_values"):
            values = ", ".join(repr(value) for value in column["all_values"])
            parts.append(f"values: {values}")
        elif column.get("examples"):
            values = ", ".join(repr(value) for value in column["examples"])
            parts.append(f"e.g. {values}")

    if column["missing"]:
        parts.append(f"{column['missing']} missing")

    return "  " + " | ".join(parts)


def describe_flag(flag):
    """Cleaning flags rewritten as warnings for the model."""

    column, issue, detail = flag["column"], flag["issue"], flag["detail"]
    detail = detail.replace("; kept.", "").replace("; not merged.", "")

    if issue == "missing_values":
        return None                      # already on the column line
    if issue == "outliers":
        return f"{column}: {detail}"
    if issue == "similar_categories":
        return f"{column}: {detail}"
    if issue == "ambiguous_dates":
        return f"{column}: dates are TEXT in an unknown day/month order"
    if issue == "partly_numeric":
        return f"{column}: stored as TEXT because some values are not numbers"
    if issue == "identifier_kept_as_text":
        return f"{column}: identifier with leading zeros, compare as TEXT"

    return f"{column}: {detail}"


def build_schema_text(profile):
    """Compact schema of every cleaned table in one dataset."""

    blocks = []

    for table_name, table in profile["tables"].items():
        lines = [f"TABLE {table_name} ({table['rows']} rows)"]

        lines += [describe_column(name, column)
                  for name, column in table["columns"].items()]

        warnings = [text for text in map(describe_flag, table["flags"])
                    if text]

        if warnings:
            lines.append("  Warnings:")
            lines += [f"  - {warning}" for warning in warnings]

        blocks.append("\n".join(lines))

    blocks.append(
        "SQL notes: SQLite dialect. Text comparisons are case-sensitive. "
        "For median / standard deviation / percentile use "
        "median_of(json_group_array(col)), stdev_of(json_group_array(col)), "
        "percentile_of(json_group_array(col), 90)."
    )

    return "\n\n".join(blocks)


schema_texts = {
    source_file: build_schema_text(profile)
    for source_file, profile in profiles.items()
}

for source_file, text in schema_texts.items():
    print(f"########## {source_file}  (~{len(text) // 4} tokens)")
    print(text, "\n")

########## customers.json  (~184 tokens)
TABLE customers (8 rows)
  customer_id INTEGER | range 1 to 8
  name TEXT | unique per row | values: 'Arjun Mehta', 'Asha Verma', 'Kavya Iyer', 'Neha Gupta', 'Priya Nair', 'Rahul Singh', 'Rohan Das', 'Vikram Rao'
  signup TEXT | date 'YYYY-MM-DD', 2023-02-22 to 2023-11-10
  tags TEXT | values: '["bulk", "vip"]', '["new", "bulk"]', '["online", "new"]', '["online", "vip"]', '["vip", "new"]', '["vip", "online"]'
  address_city TEXT | values: 'Delhi', 'Mumbai', 'Pune'
  address_pin INTEGER | range 110001 to 411001

SQL notes: SQLite dialect. Text comparisons are case-sensitive. For median / standard deviation / percentile use median_of(json_group_array(col)), stdev_of(json_group_array(col)), percentile_of(json_group_array(col), 90). 

########## messy_employees.xlsx  (~263 tokens)
TABLE messy_employees_employees (20 rows)
  emp_id TEXT | unique per row | e.g. 'E200', 'E201', 'E202'
  full_name TEXT | unique per row | e.g. 'Employee 1', 'Employee 2',

## 7. Checks

In [7]:
# ============================================================
# 7. CHECKS
# ============================================================

def query(database_path, sql, params=()):
    with sqlite3.connect(database_path) as connection:
        register_functions(connection)
        return connection.execute(sql, params).fetchall()


sales_df = datasets["messy_sales.csv"][0]["cleaned"]
sales_profile = profiles["messy_sales.csv"]["tables"]["messy_sales"]

# Row counts: cleaned 32, raw keeps all 35 original rows.
assert query(sales_db, "SELECT COUNT(*) FROM messy_sales")[0][0] == 32
assert query(sales_db, "SELECT COUNT(*) FROM messy_sales_raw")[0][0] == 35

# Column types in SQLite.
column_types = {row[1]: row[2] for row in
                query(sales_db, "PRAGMA table_info(messy_sales)")}
assert column_types["order_date"] == "TEXT"
assert column_types["units_sold"] == "INTEGER"
assert column_types["paid"] == "INTEGER"
raw_types = {row[2] for row in
             query(sales_db, "PRAGMA table_info(messy_sales_raw)")}
assert raw_types == {"TEXT"}

# Stored values: ISO dates, 1/0 booleans, NULL for missing, IDs as text.
assert query(sales_db, "SELECT MIN(order_date) FROM messy_sales")[0][0] \
    == sales_df["order_date"].min().strftime("%Y-%m-%d")
assert set(row[0] for row in
           query(sales_db, "SELECT DISTINCT paid FROM messy_sales")) == {0, 1}
assert query(sales_db,
             "SELECT COUNT(*) FROM messy_sales WHERE units_sold IS NULL")[0][0] == 3
assert query(sales_db, "SELECT order_id FROM messy_sales LIMIT 1")[0][0] == "00101"

# SQL totals match pandas.
sql_totals = dict(query(sales_db,
    "SELECT city, SUM(total_sales) FROM messy_sales GROUP BY city"))
pandas_totals = sales_df.groupby("city")["total_sales"].sum().to_dict()
assert sql_totals == {city: int(total) for city, total in pandas_totals.items()}

# Date functions work on the ISO text.
months = query(sales_db, """
    SELECT strftime('%Y-%m', order_date) AS month, COUNT(*)
    FROM messy_sales GROUP BY month ORDER BY month""")
assert [month for month, _ in months] == ["2024-01", "2024-02", "2024-03"]
assert sum(count for _, count in months) == 32

# Registered statistics match Python.
assert query(sales_db,
    "SELECT median_of(json_group_array(unit_price)) FROM messy_sales")[0][0] \
    == sales_df["unit_price"].dropna().median()
assert median_of("[]") is None and stdev_of("[5]") is None
assert percentile_of("[1, 2, 3, 4, 5]", 50) == 3

# Profile and schema text.
assert sales_profile["columns"]["units_sold"]["missing"] == 3
assert sales_profile["columns"]["city"]["all_values"] == sorted(
    sales_df["city"].dropna().unique())
assert sales_profile["columns"]["order_id"]["is_unique"]
sales_schema = schema_texts["messy_sales.csv"]
assert "'Mumbay'" in sales_schema and "messy_sales_raw" not in sales_schema
assert "percent (10 means 10%)" in sales_schema
assert "boolean stored as 1 = true" in sales_schema
assert "range 1600 to 9999999" in sales_schema            # no commas
assert "notes TEXT | values" in sales_schema              # not a key
assert "order_id TEXT | unique per row" in sales_schema

# Excel: two tables in one database.
employees_db = dataset_folder("messy_employees.xlsx") / DATABASE_NAME
tables_in_db = {row[0] for row in query(employees_db,
    "SELECT name FROM sqlite_master WHERE type = 'table'")}
assert tables_in_db == {
    "messy_employees_employees", "messy_employees_employees_raw",
    "messy_employees_targets", "messy_employees_targets_raw",
}
assert "unknown day/month order" in schema_texts["messy_employees.xlsx"]

print("All checks passed.")

All checks passed.


## 8. Save outputs

Each dataset folder now mirrors the planned API workspace:
`dataset.db`, `cleaning_report.json`, `profile.json`, `schema.txt`.

In [8]:
# ============================================================
# 8. SAVE OUTPUTS
# ============================================================

def write_json(path, data):
    with path.open("w", encoding="utf-8") as file:
        json.dump(data, file, indent=2, ensure_ascii=False, default=str)


for source_file, tables in datasets.items():
    folder = dataset_folder(source_file)

    write_json(folder / REPORT_NAME,
               [table["report"] for table in tables])
    write_json(folder / PROFILE_NAME, profiles[source_file])
    (folder / SCHEMA_NAME).write_text(schema_texts[source_file],
                                      encoding="utf-8")

for folder in sorted(DATASETS_DIR.iterdir()):
    print(f"{folder.name:<18} {sorted(path.name for path in folder.iterdir())}")

customers          ['cleaning_report.json', 'dataset.db', 'profile.json', 'schema.txt']
messy_employees    ['chats.db', 'cleaning_report.json', 'dataset.db', 'profile.json', 'schema.txt']
messy_sales        ['chats.db', 'cleaning_report.json', 'dataset.db', 'profile.json', 'schema.txt']
products           ['cleaning_report.json', 'dataset.db', 'profile.json', 'schema.txt']
